In [ ]:
# I am using LangChain to build a RAG based PDF Chatbot (a simple one)

In [ ]:
!pip install -U bitsandbytes
!pip install pipeline PyPDF2 langchain-community sentence-transformers chromadb

In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import transformers
from transformers import pipeline


from langchain.llms import HuggingFacePipeline
from PyPDF2 import PdfReader
from langchain.text_splitter import SpacyTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
model_name = "meta-llama/Llama-3.2-3B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
pline = pipeline(
    model = model,
    tokenizer = tokenizer,
    return_full_text = True,
    task = "text-generation",
    temperature = 0.7,
    max_new_tokens = 512,
    top_p = 0.95,
    repetition_penalty = 1.15
)

Device set to use cuda:0


In [ ]:
llm = HuggingFacePipeline(pipeline = pline)

<ipython-input-7-558ad789ba89>:1: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline = pline)


In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
client = chromadb.Client()
collection = client.create_collection("document1")

In [ ]:

def getchunks(text):
    splitter = SpacyTextSplitter(
        separator="\n\n",
        chunk_size=100,
        chunk_overlap=20,
        length_function=len,
    )
    return splitter.split_text(text)

In [ ]:

def load_data_convert_to_chunks(pdf):
  text = ""
  for page in pdf.pages:
    text+=page.extract_text()
  return getchunks(text)

In [ ]:
def generate_embeddings(text):
  return embedder.encode(text, convert_to_tensor=True)

In [ ]:

def store_to_db(text):

  for i, chunk in enumerate(text):
    embedding = generate_embeddings(chunk).tolist()
    collection.add(
        documents = [chunk],
        embeddings = [embedding],
        ids = [f'{i}']
    )

In [ ]:

def get_related_query(query):
  embeds = generate_embeddings(query)
  # print(query)
  # print(embeds)
  results = collection.query(
      # query_embeddings = [embeds.tolist()],
      query_texts = query,
      n_results = 3
  )
  # print(results['documents'])
  return results['documents']

In [ ]:

def get_answer(query, context):
    print("------------------------")
    # context_text = " ".join(context)  # Join if context is a list
    prompt = f"Context: {context}\n\nQuery: {query}\n\nAnswer:"
    # input = tokenizer(prompt, return_tensors='pt')
    # output = model.generate(**input, max_length=300)
    # return tokenizer.decode(output[0], skip_special_tokens=True)
    return llm.predict(prompt)

In [ ]:

def answer_query(query):
  context = (get_related_query(query))
  answer = get_answer(query, context)
  return answer

In [ ]:

def server(pdf):
  store_to_db(load_data_convert_to_chunks(pdf))
  while True:
    query = input("Enter your query: ")
    answer = answer_query(query)
    print(answer)


In [ ]:
pdf = PdfReader('/content/1. The Metamorphosis, Franz Kafka.pdf')
server(pdf)

/usr/local/lib/python3.11/dist-packages/spacy/util.py:1740: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)
/usr/local/lib/python3.11/dist-packages/spacy/pipeline/lemmatizer.py:211: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


Enter your query: Purpose of Book
Enter your query: what is the pdf about


KeyboardInterrupt: Interrupted by user